# **ML: Assignment - 7 ( Q.3 )**

Mohd Talha Patrawala

CMPN-B

23102B0025

In [3]:
import pandas as pd
import numpy as np
import argparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [4]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df = df.drop("customerID", axis=1)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df = df.dropna()

df.shape

(7032, 20)

In [6]:
for col in df.select_dtypes(include="object").columns:
    df[col] = LabelEncoder().fit_transform(df[col])

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [7]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

In [8]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [10]:
C_values = [0.1, 1, 10]

results = []
predictions_list = []

for C in C_values:

    model = SVC(kernel="linear", C=C, probability=True)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:,1]

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_score)
    n_sv = model.n_support_.sum()

    print("\nC =", C)
    print("Confusion Matrix:\n", cm)
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1:", f1)
    print("ROC-AUC:", roc)
    print("Support Vectors:", n_sv)

    results.append([C, acc, prec, rec, f1, roc, n_sv])

    temp = pd.DataFrame({
        "Actual": y_test,
        "Predicted": y_pred,
        "Score": y_score
    })

    predictions_list.append(temp)


C = 0.1
Confusion Matrix:
 [[916 117]
 [173 201]]
Accuracy: 0.7938877043354655
Precision: 0.6320754716981132
Recall: 0.5374331550802139
F1: 0.5809248554913294
ROC-AUC: 0.8281897903929678
Support Vectors: 2596

C = 1
Confusion Matrix:
 [[915 118]
 [173 201]]
Accuracy: 0.7931769722814499
Precision: 0.6300940438871473
Recall: 0.5374331550802139
F1: 0.5800865800865801
ROC-AUC: 0.8282881488422176
Support Vectors: 2586

C = 10
Confusion Matrix:
 [[915 118]
 [173 201]]
Accuracy: 0.7931769722814499
Precision: 0.6300940438871473
Recall: 0.5374331550802139
F1: 0.5800865800865801
ROC-AUC: 0.8282389696175927
Support Vectors: 2587


In [11]:
results_df = pd.DataFrame(results, columns=[
    "C","Accuracy","Precision","Recall","F1","ROC_AUC","Support_Vectors"
])

results_df

,C,Accuracy,Precision,Recall,F1,ROC_AUC,Support_Vectors
0,0.1,0.793888,0.632075,0.537433,0.580925,0.828190,2596
1,1.0,0.793177,0.630094,0.537433,0.580087,0.828288,2586
2,10.0,0.793177,0.630094,0.537433,0.580087,0.828239,2587


In [12]:
results_df.to_csv("svm_linear_results.csv", index=False)

predictions = pd.concat(predictions_list)

predictions.to_csv("test_predictions.csv", index=False)

print("Files saved successfully")

Files saved successfully


# **Interpretation: How C affects margin and overfitting/underfitting**


---


---




**C = 0.1**

* Smaller C → wider margin, allows more misclassification.

* Model is simpler and less sensitive to noise.

* Slightly more support vectors (2596) → softer margin.

* May lead to mild underfitting, but generalizes well.


---



**C = 1**

* Balanced trade-off between margin size and classification error.

* Margin slightly narrower than C = 0.1.

* Lowest support vectors (2586).

* Performance metrics remain stable.


---



**C = 10**

* Larger C → narrow margin, strongly penalizes misclassification.

* Model tries to fit training data more strictly.

* Risk of overfitting, but metrics show no real improvement.


---


# **Conclusion:**

Among the tested values, C = 1 provides the best generalization. It achieves almost the same Accuracy (0.793), F1 (0.58), and ROC-AUC (0.828) as the other values while using the lowest number of support vectors (2586). This indicates that the model maintains a good balance between margin width and classification error, avoiding both underfitting (very small C) and overfitting (very large C). Therefore, C = 1 is the most suitable choice for this churn prediction model.
